# Problem D: Prediction of Traffic Flow based on Burgers’ Equation Model

Burgers' equation is a classical PDE model that captures the interplay between nonlinear convection and diffusion, making it well-suited for describing traffic flow dynamics. In this context, the convective term models the tendency for traffic to compress as vehicle density increases, while the diffusive term reflects the smoothing effect of individual driver reactions to surrounding traffic conditions.

The viscous Burgers' equation governing the evolution of vehicle velocity is:

$$
u_t + uu_x = \nu u_{xx}, \qquad x\in(-1,1),\ t\in(0,1] \tag{3}
$$

where:
- $u(x,t)$: vehicle velocity field (m/s),
- $\nu = 0.1/\pi$: kinematic viscosity coefficient (a higher value reflects more cautious, diffusive driver behavior),
- $x$: position along the road segment,
- $t$: time.

Homogeneous Dirichlet boundary conditions are imposed at both ends of the road:

$$
u(-1,\, t) = u(1,\, t) = 0, \qquad t \in (0, 1]
$$

Given an initial velocity profile $u(x, 0) = a(x)$, the full spatio-temporal velocity field $u(x,t)$ for $t > 0$ is uniquely determined by solving Eq. (3). In practice, however, running a high-fidelity finite difference solver for every new initial condition is computationally prohibitive. The goal of this task is therefore to learn a fast surrogate that approximates the solution operator $a(x) \mapsto u(x,t)$.

## Task: predicting the velocity filed $u(x,t)$ given the inital field $a(x)$ 

A dataset of initial conditions $a(x)$ sampled from a distribution $\mathcal{A}$ has been collected, along with the corresponding velocity fields $u(x,t)$ computed by a high-precision Finite Difference Method (FDM) solver. The dataset is **partially labeled**: only a small subset of initial conditions is paired with its FDM solution, while a much larger set of initial conditions has no corresponding solution available. This reflects a realistic scenario in which high-fidelity simulation data is scarce and expensive to obtain. The trained surrogate model should accurately predict $u(x,t)$ for any new initial condition $a(x)$ drawn from the same distribution $\mathcal{A}$, at negligible computational cost.

### Goals

- Select a suitable deep learning method for this task and justify your choice. 
- Report your implementation setup, including network architecture, activation function, optimizer (with learning rate), number of epochs, batch size, loss formulation, and any additional techniques used for improvement.
- Compute the $L^2$ relative error on the **test dataset** at each training epoch and plot the `Error vs. Epoch` curve (reporting the final error). The error is defined as:

$$
\text{error} = \frac{1}{N}\sum_{j=1}^{N}\sqrt{\frac{\displaystyle\sum_i\left|u^{(j)}_{\text{pred}}(x_i,t_i)-u^{(j)}_{\text{true}}(x_i,t_i)\right|^2}{\displaystyle\sum_i\left|u^{(j)}_{\text{true}}(x_i,t_i)\right|^2}}
$$

where $j$ indexes the sample and $i$ indexes the spatio-temporal grid point.

- For the **first test instance**, use `matplotlib` to produce four separate figures:
  1. The initial condition $a(x) = u(x, 0)$
  2. The predicted velocity field $u_{\text{pred}}(x,t)$
  3. The ground truth velocity field $u_{\text{true}}(x,t)$
  4. The pointwise absolute error $|u_{\text{pred}} - u_{\text{true}}|$

### Dataset

All data are provided in `ProblemD_dataset.h5`:

| Key | Shape | Description |
|---|---|---|
| `a_train_labeled` | (200, 256) | $N=200$ **labeled** initial fields, sampled at 256 spatial sensors |
| `u_train_labeled` | (200, 200, 256) | Corresponding FDM-computed velocity fields on a $200\times256$ spatio-temporal grid |
| `a_train_unlabeled` | (1800, 256) | $N=1800$ **unlabeled** initial fields (no paired solution available) |
| `a_test` | (200, 256) | Test initial conditions (**do not use for training**) |
| `u_test` | (200, 200, 256) | Ground truth velocity fields for test instances (**do not use for training**) |
| `x_mesh` | (256, 1) | Spatial coordinates of the 256 grid points |
| `t_mesh` | (200, 1) | Temporal coordinates of the 200 time steps |

> 📥 Dataset download: [https://www.kaggle.com/datasets/yhzang32/dno4pdes](https://www.kaggle.com/datasets/yhzang32/dno4pdes)

In [1]:
import os,time
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"  #To avoid dying kernel issue this has been used in each excercise
import numpy as np
import h5py
import torch
import matplotlib;matplotlib.use('Agg')
import matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)
dtype = torch.float32
######################################
# Load training data
######################################
with h5py.File('ProblemD_dataset.h5', 'r') as file:
    print(file.keys())
    t_mesh = torch.tensor(np.array(file['t_mesh']), dtype=dtype)
    x_mesh = torch.tensor(np.array(file['x_mesh']), dtype=dtype)
    a_test = torch.tensor(np.array(file['a_test']), dtype=dtype)
    u_test = torch.tensor(np.array(file['u_test']), dtype=dtype)
    a_train = torch.tensor(np.array(file['a_train_labeled']), dtype=dtype)
    u_train = torch.tensor(np.array(file['u_train_labeled']), dtype=dtype)

   # X, T = np.meshgrid(x_mesh, t_mesh)
#
#print('t_mesh:', t_mesh.shape, 'x_mesh:', x_mesh.shape)
#print('a_train_labeled:', a_train_labeled.shape, 'u_train_labeled:', u_train_labeled.shape)
#print('a_train_unlabeled:', a_train_unlabeled.shape)
#print('a_test:', a_test.shape, 'u_test:', u_test.shape)
#################################
#inx = 0
#fig, axes = plt.subplots(1,2, figsize=(16,4))
#
#cntr = axes[0].plot(x_mesh, a_train[0])
#axes[0].set_title('The initial field a(x)')
#
#cntr = axes[1].contourf(T, X, u_train[inx])
#axes[1].set_title('The velocity field u(t,x)')
#plt.colorbar(cntr)
#
#plt.show()

<KeysViewHDF5 ['a_test', 'a_train_labeled', 'a_train_unlabeled', 't_mesh', 'u_test', 'u_train_labeled', 'x_mesh']>


In [2]:
#Burgers Surogate Model
class BurgersSurrogate2(torch.nn.Module):
    #Encoder-decoder surrogate a(x)-u(x,t).
    #No mean pooling to preserve spatial relationship
    #ConvTranspose2d decoder generates the full 200x256 spatio-temporal field.
    def __init__(self):
        super().__init__()
        self.enc = torch.nn.Sequential(
            torch.nn.Conv1d(1, 32, 5, stride=2, padding=2), torch.nn.ReLU(),   # 256->128
            torch.nn.Conv1d(32, 64, 5, stride=2, padding=2), torch.nn.ReLU(),  # 128->64
            torch.nn.Conv1d(64, 128, 5, stride=2, padding=2), torch.nn.ReLU()) # 64->32
        self.fc = torch.nn.Sequential(
            torch.nn.Linear(128 * 32, 1024), torch.nn.ReLU(),
            torch.nn.Linear(1024, 128 * 25 * 32), torch.nn.ReLU())
        self.dec = torch.nn.Sequential(
            torch.nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), torch.nn.ReLU(),  # 25x32->50x64
            torch.nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1), torch.nn.ReLU(),   # ->100x128
            torch.nn.ConvTranspose2d(32, 16, 4, stride=2, padding=1), torch.nn.ReLU(),   # ->200x256
            torch.nn.Conv2d(16, 1, 3, padding=1))
    def forward(self, a):
        B = a.shape[0]
        z = self.enc(a.unsqueeze(1)).reshape(B, -1)
        z = self.fc(z).reshape(B, 128, 25, 32)
        return self.dec(z).squeeze(1)  # (B,200,256)


In [3]:
#Training
model = BurgersSurrogate2()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
sched = torch.optim.lr_scheduler.StepLR(opt, step_size=25, gamma=0.5)
epochs, batch_size = 60, 20
errors=[]; t0=time.time()
for ep in range(epochs):
    perm = torch.randperm(a_train.shape[0])
    a_train, u_train = a_train[perm], u_train[perm]
    for i in range(0, a_train.shape[0], batch_size):
        opt.zero_grad()
        loss = ((model(a_train[i:i+batch_size])-u_train[i:i+batch_size])**2).mean()
        loss.backward(); opt.step()
    sched.step()
    with torch.no_grad():
        preds=[model(a_test[i:i+50]) for i in range(0,200,50)]
        u_pred_test=torch.cat(preds)
        err=torch.mean(torch.sqrt(((u_pred_test-u_test)**2).sum((1,2))/(u_test**2).sum((1,2)))).item()
        errors.append(err)
    print(f"ep {ep} loss {loss.item():.4e} err {err:.4f} ({time.time()-t0:.0f}s)", flush=True)

print("FINAL_ERROR", errors[-1])
np.save('D_errors.npy', np.array(errors))
torch.save(model.state_dict(), 'D_model.pt')

ep 0 loss 1.8138e-02 err 0.9710 (4s)
ep 1 loss 1.1695e-02 err 0.7075 (7s)
ep 2 loss 6.9319e-03 err 0.5938 (11s)
ep 3 loss 3.1715e-03 err 0.4299 (14s)
ep 4 loss 8.8911e-04 err 0.3735 (18s)
ep 5 loss 1.0802e-03 err 0.2929 (21s)
ep 6 loss 4.5836e-04 err 0.2214 (25s)
ep 7 loss 2.9421e-04 err 0.2132 (28s)
ep 8 loss 2.7047e-04 err 0.1784 (32s)
ep 9 loss 1.8297e-04 err 0.1631 (35s)
ep 10 loss 1.9473e-04 err 0.1523 (39s)
ep 11 loss 1.4927e-04 err 0.1587 (42s)
ep 12 loss 9.2015e-05 err 0.1266 (46s)
ep 13 loss 1.0403e-04 err 0.1200 (49s)
ep 14 loss 1.0298e-04 err 0.1115 (53s)
ep 15 loss 7.4477e-05 err 0.1093 (56s)
ep 16 loss 9.6029e-05 err 0.1035 (60s)
ep 17 loss 7.3430e-05 err 0.1035 (64s)
ep 18 loss 6.4070e-05 err 0.0980 (67s)
ep 19 loss 6.3045e-05 err 0.1032 (71s)
ep 20 loss 6.0130e-05 err 0.1010 (74s)
ep 21 loss 8.0206e-05 err 0.1045 (78s)
ep 22 loss 9.5362e-05 err 0.1256 (81s)
ep 23 loss 8.6209e-05 err 0.1074 (85s)
ep 24 loss 5.3745e-05 err 0.0970 (89s)
ep 25 loss 6.4826e-05 err 0.0916 (92s

In [4]:
#Plots
fig_dir='.'; os.makedirs(fig_dir, exist_ok=True)# To avoid dying kernel issue
plt.figure(figsize=(6,4))
plt.plot(errors)
plt.xlabel("Epoch")
plt.ylabel("Relative $L^2$ error"); plt.grid(True)
plt.tight_layout()
plt.savefig(f'{fig_dir}/D_error_vs_epoch.png', dpi=150); plt.close()
a0=a_test[0].numpy()
u_true0=u_test[0].numpy()
u_pred0=u_pred_test[0].numpy()
err0=np.abs(u_pred0-u_true0)
x_np=x_mesh.numpy().flatten()
t_np=t_mesh.numpy().flatten()
extent=[x_np.min(),x_np.max(),t_np.min(),t_np.max()]
plt.figure(figsize=(5,4))
plt.plot(x_np,a0); plt.xlabel('$x$')
plt.ylabel('$a(x)$')
plt.title('Initial condition $a(x)=u(x,0)$')
plt.grid(True)
plt.tight_layout()
plt.savefig(f'{fig_dir}/D_a0.png', dpi=150); plt.close()
for arr,name,title in [(u_pred0,'D_u_pred','Predicted $u(x,t)$'),
                       (u_true0,'D_u_true','True $u(x,t)$'),
                       (err0,'D_u_abs_err','Pointwise absolute error')]:
    plt.figure(figsize=(5,4))
    im=plt.imshow(arr,aspect='auto',origin='lower',extent=extent)
    plt.colorbar(im); plt.title(title); plt.xlabel('$x$'); plt.ylabel('$t$')
    plt.tight_layout(); plt.savefig(f'{fig_dir}/{name}.png', dpi=150); plt.close()
print("done")

done
